<img src="https://hilpisch.com/tpq_logo_bic.png" width="320" align="right">

# Reinforcement Learning for Dynamic Asset Allocation

## Deep-Q Network (DQN) Agent Workbench

Dr. Yves J. Hilpisch | https://tpq.io

This notebook presents the value-based discrete agent as a compact result story. The default configuration uses Double Deep Q-Learning, while regular Deep Q-Learning can be tested by setting `double_dqn: false`.

## Configuration

The experiment profile is read from the sweep-derived YAML configuration used by the DQN run.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

CONFIG_PATH = (ROOT / 'configs' / 'dqn_best.yml') if (ROOT / 'configs' / 'dqn_best.yml').exists() else (ROOT / 'configs' / 'dqn_ref.yml')
TRAIN_SLICES_OVERRIDE = None
DEFAULT_BASELINE_NAME = 'grid_best'
DEFAULT_TEST_SLICE = 0
EVAL_SAMPLING_SEED = 4444
EVAL_TEST_SLICES_OVERRIDE = 10
TRAINING_DEVICE = 'cpu'

In [ ]:
%config InlineBackend.figure_format = 'svg'

## Runtime Setup

The runtime setup imports the notebook helpers and resolves the project root. These helpers keep training and result handling in the package.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, Markdown, display

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

The following import block loads the functions used for configuration, DQN training, comparison, and visual inspection.

In [ ]:
from rl4am.notebooks.dqn_workbench import (
    apply_dqn_training_overrides,
    build_comparison_artifacts,
    build_dqn_config_overview_frame,
    build_dqn_training_summary_frame,
    build_sweep_winner_frame,
    build_publication_figure,
    build_runtime_overview_frame,
    build_sampling_overview_frame,
    build_selected_slice_story,
    compact_metric_frame,
    configure_notebook_display,
    load_dqn_workbench_config,
    plot_dqn_training_history,
    plot_visual_diagnostics,
    prepare_dqn_context,
    resolve_project_root,
    run_baselines,
    run_dqn_training,
)

configure_notebook_display()
ROOT = resolve_project_root(Path.cwd())

The runtime constants select the sweep-derived configuration, optional training-slice override, comparison baseline, and selected test slice.

The context resolves the YAML configuration, loads market data, samples training slices, optionally samples a separate evaluation draw, and creates the run directories.

In [ ]:
config, config_notes = load_dqn_workbench_config(config_path=CONFIG_PATH)
config = apply_dqn_training_overrides(
    config,
    train_slices=TRAIN_SLICES_OVERRIDE,
)
context = prepare_dqn_context(
    root=ROOT,
    config_path=CONFIG_PATH,
    config=config,
    config_notes=config_notes,
    selected_test_slice=DEFAULT_TEST_SLICE,
    evaluation_seed=EVAL_SAMPLING_SEED,
    evaluation_test_slices=EVAL_TEST_SLICES_OVERRIDE,
    baseline_name=DEFAULT_BASELINE_NAME,
)

The DQN configuration overview summarises the market, model, replay, target-network, and exploration settings.

In [ ]:
display(build_dqn_config_overview_frame(context))

The runtime overview records where the run artefacts and comparison outputs are written.

In [ ]:
display(
    build_runtime_overview_frame(context).query(
        "item != 'selected_test_slice'"
    )
)

The sampling overview summarises the data span and the train/test slicing plan.

In [ ]:
display(build_sampling_overview_frame(context))

If the configuration loader adjusted any notebook-facing fields, the notes below record those adjustments.

In [ ]:
if context.config_notes:
    display(Markdown("\n".join(
        f"- {note}" for note in context.config_notes
    )))

## Aggregate Baselines

The deterministic baselines are calibrated on each training slice, averaged into one fixed weight per strategy, and then evaluated on the full test-slice set. Individual slice details are reserved for the final inspection section.

In [ ]:
baselines = run_baselines(context)

The baseline setup table records the deterministic strategy parameters used in the comparison.

In [ ]:
display(baselines.baseline_setup)

The baseline summary aggregates performance across the sampled test slices.

In [ ]:
display(baselines.baseline_slice_summary.set_index(['strategy', 'stat']))

The diagnostic note below appears only when the baseline weights collapse to a common boundary or value.

In [ ]:
if baselines.boundary_note:
    display(Markdown(baselines.boundary_note))

## DQN Training

The DQN agent collects transitions from sampled training slices, stores them in replay memory, and updates the online network from mini-batches. Double DQN target selection is active when `double_dqn` is true.

In [ ]:
dqn = run_dqn_training(context, device=TRAINING_DEVICE)

The training summary compares the first and final update diagnostics.

In [ ]:
display(build_dqn_training_summary_frame(dqn))

The final rows of the training history show the latest reward, loss, replay, and exploration diagnostics.

In [ ]:
display(dqn.history_frame.tail())

The training-history plot visualises reward, temporal-difference loss, and exploration over updates.

In [ ]:
plot_dqn_training_history(dqn.history_frame)

The policy summary aggregates the greedy DQN policy performance across all test slices.

In [ ]:
display(dqn.slice_summary.set_index(['strategy', 'stat']))

The saved run path identifies the canonical result directory used by the comparison layer.

In [ ]:
display(Markdown(f'Saved DQN run to `{context.policy_dir}`.'))

## Aggregate Comparison

The comparison artefacts reload saved result files and compare the DQN policy with the selected baseline across the full test-slice set.

In [ ]:
comparison = build_comparison_artifacts(context, baselines, dqn)

The following table summarises metric differences between the policy and both benchmark references across all test slices.

In [ ]:
display(comparison.diff_summary)

The win-rate table reports how often the policy outperforms each benchmark by metric across the test-slice set.

In [ ]:
print(comparison.win_summary)

The best-slice table ranks the strongest DQN test slices by terminal equity and annualised Sharpe.

In [ ]:
display(comparison.best_test_slices)

The sweep winner table lists the strongest successful DQN sweep runs when sweep artifacts are available.

In [ ]:
display(build_sweep_winner_frame(ROOT, sweep_dir='runs/dqn_sweep').T)

## Selected Slice Inspection

This final section selects one test slice for detailed inspection. Change `INSPECT_TEST_SLICE` or `INSPECT_BASELINE_NAME` to inspect another saved slice without rerunning training.

In [ ]:
INSPECT_TEST_SLICE = 9
INSPECT_BASELINE_NAME = 'grid_best'

The inspection context points the comparison helpers at the chosen test slice and baseline.

In [ ]:
inspection_context = prepare_dqn_context(
    root=ROOT,
    config_path=CONFIG_PATH,
    config=config,
    config_notes=config_notes,
    evaluation_test_slices=EVAL_TEST_SLICES_OVERRIDE,
    selected_test_slice=INSPECT_TEST_SLICE,
    baseline_name=INSPECT_BASELINE_NAME,
)
inspection = build_comparison_artifacts(
    inspection_context,
    baselines,
    dqn,
)

The selected-slice manifest identifies the evaluation window used for the detailed inspection.

In [ ]:
selected_slice = inspection_context.eval_slices[
    inspection_context.selected_test_slice
]
display({
    'slice_id': selected_slice.slice_id,
    'split': selected_slice.split,
    'start_date': selected_slice.start_date,
    'end_date': selected_slice.end_date,
    'rows': int(selected_slice.returns.shape[0]),
})

The selected-slice story table compares the policy and baseline on the main metrics for the chosen window.

In [ ]:
display(build_selected_slice_story(
    inspection,
    inspection_context.baseline_name,
))

The compact metric table provides the same selected-slice comparison in the standard metric layout.

In [ ]:
display(compact_metric_frame(inspection.selected_metrics))

The path diagnostics show net equity, turnover, and risky allocation for the selected slice.

In [ ]:
plot_visual_diagnostics(inspection)

The report figure displays the saved equity comparison for the selected slice.

In [ ]:
inspection_figure_path = build_publication_figure(
    inspection,
    inspection_context,
)

The output path below records where the selected-slice comparison assets are stored.

In [ ]:
display(Markdown(
    f'Comparison assets written to `{inspection_context.selected_report_dir}`.'
))

The saved figure is shown below for direct visual inspection.

In [ ]:
display(Image(filename=str(inspection_figure_path), width=760))

<img src="https://hilpisch.com/tpq_logo_bic.png" width="320" align="right">